In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
test = pd.read_csv('/kaggle/input/competitions/triagegeist/test.csv')

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df['site_id'].unique()

In [ ]:
# encoding side_id
site = {
    'SITE-TMP-01' : 1,
    'SITE-HEL-01' : 2,
    'SITE-HEL-02' : 3,
    'SITE-TUR-01' : 4,
    'SITE-OUL-01' : 5
}

df['site_id'] = df['site_id'].map(site)

In [ ]:
df['triage_nurse_id'].unique()

In [ ]:
## frequency encoding the nurses ,  because behavior of nurses is a key feature for acuity
freq = df['triage_nurse_id'].value_counts().to_dict()
df['triage_nurse_id'] = df['triage_nurse_id'].map(freq)

In [ ]:
df.head()

In [ ]:
df['arrival_mode'].unique()

In [ ]:
# manual encoding arrival_mode
arrival = {
    'walk-in':1,
    'police':2,
    'ambulance':3,
    'transfer':4,
    'helicopter':5,
    'brought_by_family':6,
}
df['arrival_mode'] = df['arrival_mode'].map(arrival)

In [ ]:
df.head()

In [ ]:
df['arrival_day'].unique()

In [ ]:
## Cyclic encoding for our week days

day_map = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

df['day_num'] = df['arrival_day'].map(day_map)

In [ ]:
## Applying sin , cos transformations

df['day_sin'] = np.sin(2 * np.pi * df['day_num'] / 7)
df['day_cos'] = np.cos(2 * np.pi * df['day_num'] / 7)

In [ ]:
df['day_sin']

In [ ]:
df['day_cos']

In [ ]:
### why we did cyclic encoding instead of manual encoding , becoz in that case model thinks sunday=0 is > saturday=6

In [ ]:
# now we remove df['arrival_day] and day_num

df = df.drop(['arrival_day' , 'day_num'] , axis = 1)

In [ ]:
df['arrival_season'].unique()

In [ ]:
# checking behavior of seasons with target (triage_acuity)

df.groupby('arrival_season')['triage_acuity'].mean().sort_values()

In [ ]:
sns.countplot(x = 'arrival_season' , hue = 'triage_acuity' , data = df)
plt.title('Acuity across Seasons')
plt.show()

In [ ]:
# we see all seasons are identical , so we drop this feature
df = df.drop('arrival_season' , axis = 1)

In [ ]:
df.head()
df.columns

In [ ]:
# checking monthly behavior with target feature
sns.countplot(x = 'arrival_month' , hue = 'triage_acuity' , data = df)
plt.title('Acuity across Month')
plt.show()

In [ ]:
# so we drop thius feature too
df = df.drop('arrival_month' , axis=1)

In [ ]:
### encoding done till arrival_mode , 
# have to start from arrival_hour

df.head()

In [ ]:
sns.countplot(x = 'arrival_hour' , hue = 'triage_acuity' , data = df)
plt.title('Acuity across Shift')
plt.show()

In [ ]:
df = df.drop('arrival_hour' , axis = 1)

In [ ]:
# checking arrival shift distribution
sns.countplot(x = 'shift' , hue = 'triage_acuity' , data = df)
plt.title('Acuity across Shift')
plt.show()

In [ ]:
## shift also follows same pattern eevrywhere , traige = 3 is highest
# so we drop it
df = df.drop('shift' , axis = 1)

In [ ]:
### observing age group
num_age_groups = df['age_group'].value_counts()
num_age_groups

In [ ]:
index_age_groups = df['age_group'].value_counts().index
index_age_groups

In [ ]:
plt.pie(
    num_age_groups , 
    labels = index_age_groups , 
    autopct = '%1.2f%%' , 
    startangle=90
)
plt.title('Age Group Distribution')
plt.show()

In [ ]:
sns.countplot(x = 'age_group' , hue = 'triage_acuity' , data = df)
plt.title('Acuity across Age Group')
plt.show()

In [ ]:
## encoding age group
df['age_group'] = df['age_group'].map({
    'pediatric' : 0,
    'young_adult' : 1,
    'middle_aged' : 2,
    'elderly' : 3
})

In [ ]:
# cHECKING language
df['language'].unique()

In [ ]:
num_lang = df['language'].value_counts()
num_lang

index_lang = df['language'].value_counts().index
index_lang


plt.pie(
    num_lang , 
    labels = index_lang , 
    autopct = '%1.2f%%' , 
    startangle=90
)
plt.title('Language Distribution of Pie Chart')
plt.show()

In [ ]:
sns.countplot(x = 'language' , hue = 'triage_acuity' , data = df)
plt.title('Acuity across Language')
plt.show()

In [ ]:
# %age plot for language column
pd.crosstab(df['language'] , df['triage_acuity'] , normalize = 'index')

In [ ]:
# removing language column
df = df.drop('language' , axis=1)
df.head()

In [ ]:
## Boxplot of sex with triage acuity
sns.boxplot(x = 'sex' , y = 'triage_acuity' , data = df)
plt.title('BoxPLot')
plt.show()

In [ ]:
## Checking the percentage plot of sex column
pd.crosstab(df['sex'] , df['triage_acuity'] , normalize = 'index')

In [ ]:
# dropping sex column as it's acuity is same for male and female
df = df.drop('sex' , axis = 1)

In [ ]:
df.head()

In [ ]:
##### a function of top 3 visualizattttion techniques of a column to save our time

def top3visuals(df , col , target = 'triage_acuity'):
    plt.figure(figsize = (18 , 12))  # width = 18 , height = 12

    # countplot
    plt.subplot(2 , 3 , 1)  # grid = 2 rows * 3 columns ,  postion = 1
    sns.countplot(x = col ,hue = target ,  data = df)
    plt.title(f" countplot distribution of {col} vs {target}")
    plt.xticks(rotation = 45) # for rotaing readability

    # %age plot
    plt.subplot(2 , 3 , 2)   # grid = 2 rows * 3 columns ,  postion = 2
    percent = pd.crosstab(df[col] , df[target] , normalize = 'index')# normalize='index' means we want = row-wise percentage
    # perecnt = is a daatframe of percentages
    percent.plot(kind = 'bar' , stacked = True , ax = plt.gca())  # we are now plotting this dataframe called percent , and kind = 'bar' means vertical bar chart , stacked = 'True' means each bar is on top of other making combined %age as 100% and plt.gca() = “get current axis”
    plt.title(f"Percentage Plot for {col} v/s {target}")
    plt.xticks(rotation = 45)

    # pie chart
    plt.subplot(2 , 3 , 3)   # grid = 2 rows * 3 columns ,  postion = 3
    values = df[col].value_counts()
    index = df[col].value_counts().index
    plt.pie(values , labels = index , startangle=90 , autopct = '%1.2f%%')
    plt.title(f"PIE CHART DISTRIBUTION of {col}")

    plt.tight_layout()
    plt.show()

In [ ]:
top3visuals(df , 'insurance_type')

In [ ]:
# in the above figure we see acuity 3 is dominating everwhere , so 'insurance_type' is a weak feature
# dropping it
df = df.drop('insurance_type' , axis = 1)

In [ ]:
# checking transport orgin column
top3visuals(df , 'transport_origin')

In [ ]:
df = df.drop('transport_origin' , axis = 1)

In [ ]:
# checking pain location
top3visuals(df , 'pain_location')

In [ ]:
df['pain_location'].unique()

In [ ]:
### as this is not a real dataset , so above distribution follows a similar patter , but still we dont remove it , because pain location is an important factor for triage acuity

# encoding pain location using one hot encoding

# first merging unknown and none
df['pain_location'] = df['pain_location'].replace({
    'unknown': 'other',
    'none': 'other'
})

df = pd.get_dummies(df , columns = ['pain_location'] , drop_first = True)

In [ ]:
# coming on mental_status_triage
top3visuals(df , 'mental_status_triage')

In [ ]:
## Mental status triage is a very usewfull feature fpr determining triage acuity , so we keep it.

# alert < drowsy < confused < agitated < unresponsive

# ordinal encoding it
mental_status = {
    'alert' : 0,
    'drowsy' : 1,
    'confused' : 2,
    'agitated' : 3,
    'unresponsive' : 4
}
df['mental_status_triage'] = df['mental_status_triage'].map(mental_status)

In [ ]:
df.head()

In [ ]:
# chief complaint system
top3visuals(df , 'chief_complaint_system')

# chief complaint system is an imp. feature irrespective of its distribution

In [ ]:
# 3 grouping imp ones to avoid noise

df['chief_complaint_system'] = df['chief_complaint_system'].replace({
    'neurological' : 'critical',
    'trauma' : 'critical',
    'respiratory' : 'critical',
    'cardiovascular' : 'crticial',


    'gastrointestinal' : 'moderate',
    'infectious' : 'moderate',
    'endocrine' : 'moderate',

    'dermatological' : 'mild',
    'ophthalmic' : 'mild',
    'ENT' : 'mild',

    'psychiatric' : 'special',
    'genitourunary' : 'special',
    'musculoskeletal' : 'special',
    'other' : 'special'

})

In [ ]:
df = pd.get_dummies(df , columns = ['chief_complaint_system'] , drop_first = True)

In [ ]:
df.columns

In [ ]:
cat_cols = df.select_dtypes(include = ['object' , 'string' , 'category']).columns
cat_cols

In [ ]:
df['disposition'].unique()

In [ ]:
top3visuals(df , 'disposition')

In [ ]:
### So we will encode this feature using target encoding 9 gives the average value of target variable instead of random numbers)

import category_encoders as ce

encoder = ce.TargetEncoder(cols = ['disposition'])
df['disposition'] = encoder.fit_transform(
    df['disposition'],
    df['triage_acuity']
)

In [ ]:
df[['disposition', 'triage_acuity']].head()

In [ ]:
pd.set_option('display.max_columns' , None)

In [ ]:
df.head(10)

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

In [ ]:
cols = [
    'systolic_bp',
    'diastolic_bp',
    'mean_arterial_pressure',
    'pulse_pressure',
    'respiratory_rate',
    'temperature_c',
    'shock_index',
    'triage_acuity'
]

df_selected = df[cols]

df_selected.head()

In [ ]:
for col in cols:
    df[col + '_missing'] = df[col].isna().astype(int)

In [ ]:
for col in cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df.isna().sum().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df = df.drop(['triage_acuity_missing' , 'disposition' , 'ed_los_hours'] , axis = True)

In [ ]:
df.head()

In [ ]:
df.duplicated().sum()  # means no duplicate value

In [ ]:
df.to_csv('training_set.csv' , index = False)

****Now Coming to Test Data****

In [ ]:
test.head()

In [ ]:
# encoding side_id
site = {
    'SITE-TMP-01' : 1,
    'SITE-HEL-01' : 2,
    'SITE-HEL-02' : 3,
    'SITE-TUR-01' : 4,
    'SITE-OUL-01' : 5
}

test['site_id'] = test['site_id'].map(site)

In [ ]:
freq = test['triage_nurse_id'].value_counts().to_dict()
test['triage_nurse_id'] = test['triage_nurse_id'].map(freq)

In [ ]:
arrival_test = {
    'walk-in':1,
    'police':2,
    'ambulance':3,
    'transfer':4,
    'helicopter':5,
    'brought_by_family':6,
}
test['arrival_mode'] = test['arrival_mode'].map(arrival_test)

In [ ]:
day_map_test = {
    'Monday': 0,
    'Tuesday': 1,
    'Wednesday': 2,
    'Thursday': 3,
    'Friday': 4,
    'Saturday': 5,
    'Sunday': 6
}

test['day_num'] = test['arrival_day'].map(day_map_test)

In [ ]:

test['day_sin'] = np.sin(2 * np.pi * test['day_num'] / 7)
test['day_cos'] = np.cos(2 * np.pi * test['day_num'] / 7)

In [ ]:
test = test.drop(['arrival_day' , 'day_num'] , axis = 1)
test = test.drop('arrival_season' , axis = 1)
test = test.drop('arrival_month' , axis=1)
test = test.drop('arrival_hour' , axis = 1)
test = test.drop('shift' , axis = 1)

In [ ]:
test['age_group'] = test['age_group'].map({
    'pediatric' : 0,
    'young_adult' : 1,
    'middle_aged' : 2,
    'elderly' : 3
})

In [ ]:
test = test.drop('language' , axis=1)
test = test.drop('sex' , axis = 1)
test = test.drop('insurance_type' , axis = 1)
test = test.drop('transport_origin' , axis = 1)



test['pain_location'] = test['pain_location'].replace({
    'unknown': 'other',
    'none': 'other'
})

test = pd.get_dummies(test , columns = ['pain_location'] , drop_first = True)

In [ ]:
mental_status = {
    'alert' : 0,
    'drowsy' : 1,
    'confused' : 2,
    'agitated' : 3,
    'unresponsive' : 4
}
test['mental_status_triage'] = test['mental_status_triage'].map(mental_status)

In [ ]:
test['chief_complaint_system'] = test['chief_complaint_system'].replace({
    'neurological' : 'critical',
    'trauma' : 'critical',
    'respiratory' : 'critical',
    'cardiovascular' : 'crticial',


    'gastrointestinal' : 'moderate',
    'infectious' : 'moderate',
    'endocrine' : 'moderate',

    'dermatological' : 'mild',
    'ophthalmic' : 'mild',
    'ENT' : 'mild',

    'psychiatric' : 'special',
    'genitourunary' : 'special',
    'musculoskeletal' : 'special',
    'other' : 'special'

})

In [ ]:
test = pd.get_dummies(test , columns = ['chief_complaint_system'] , drop_first = True)

In [ ]:
cols_test = [
    'systolic_bp',
    'diastolic_bp',
    'mean_arterial_pressure',
    'pulse_pressure',
    'respiratory_rate',
    'temperature_c',
    'shock_index'
]

test_selected = test[cols_test]

test_selected.head()

In [ ]:
for col in cols_test:
    test[col + '_missing'] = test[col].isna().astype(int)

In [ ]:
for col in cols_test:
    test[col] = test[col].fillna(test[col].median())

In [ ]:
test.to_csv('test_set.csv' , index = False)

In [ ]:
train = pd.read_csv('/kaggle/working/training_set.csv')
train.head()

In [ ]:
X = train.drop(['patient_id' , 'triage_acuity'] , axis = 1)
y = train['triage_acuity']

In [ ]:
test = pd.read_csv('/kaggle/working/test_set.csv')
test.head()

In [ ]:
## Splitting into training and validation data

from sklearn.model_selection import train_test_split
X_train , X_val , y_train , y_val = train_test_split(X , y , test_size = 0.20 ,stratify = y , random_state = 42)

In [ ]:
print(sorted(y_train.unique()))

In [ ]:
y_train = y_train - 1
y_val = y_val - 1

In [ ]:
X_test = test.drop('patient_id' , axis = 1)

In [ ]:
!pip install lightgbm

In [ ]:
class_weight_dict = {
    0: 1.0,
    1: 1.0,
    2: 1.2,
    3: 1.6,
    4: 2.0
}

sample_weights = y_train.map(class_weight_dict)

In [ ]:
import lightgbm as light

In [ ]:
# Creating Training Dataset
train_data = light.Dataset(X_train , label = y_train , weight = sample_weights)

val_data = light.Dataset(X_val , label =y_val)

In [ ]:
params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',

    'learning_rate': 0.02,
    'num_leaves': 100,
    'max_depth': 10,
    'min_data_in_leaf': 50,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 5,

    'lambda_l1': 0.5,
    'lambda_l2': 0.5
}

*****Now i will train my model using LightGBM Classifier , which is fast and efficient for medical and large data*****

In [ ]:
# Training our Model

model = light.train(
    params,
    train_data,
    valid_sets = [val_data] ,
    num_boost_round = 1000 ,
    callbacks = [light.early_stopping(100)]
)

In [ ]:
y_val_pred = model.predict(X_val)
y_val_pred = y_val_pred.argmax(axis=1)

from sklearn.metrics import classification_report
print(classification_report(y_val, y_val_pred))

In [ ]:
y_test_pred = model.predict(X_test)
y_test_pred = y_test_pred.argmax(axis=1)

In [ ]:
y_test_pred = y_test_pred + 1

In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_val, y_val_pred))

In [ ]:
### Now creating submission file

submission = pd.DataFrame({
    'patient_id': test['patient_id'],
    'triage_acuity': y_test_pred
})

In [ ]:
submission.to_csv("submission.csv", index=False)